# 05 — Per-Creator Models (B3, B4)

One RF and one XGBoost model per creator.

Key design choices:
- `*_vs_creator` features are **dropped** for per-creator training — within one creator
  they equal the base feature minus a constant, carrying no additional signal.
- One conservative hyperparameter config is found via CV on 100 sampled creators,
  then applied to all 1437 creators.
- Compared against B2 (global XGB) as the primary baseline.

In [ ]:
import os, time, warnings
from pathlib import Path

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")

from creator_analytics import features as fe
from creator_analytics.config import load_config
from creator_analytics.models import (
    VS_CREATOR_SUFFIX, prepare_X, make_cv_splits,
    search_per_creator_config, train_per_creator_models,
    TrainedModel,
)
from creator_analytics.evaluate import (
    compute_metrics, compute_per_creator_metrics,
    paired_comparison, log_experiment, leakage_check, save_metrics,
    predict_per_creator,
)

cfg       = load_config("config/default.yaml")
PROCESSED = Path(cfg["paths"]["processed_dir"])
PREDS_DIR = Path("results/predictions"); PREDS_DIR.mkdir(parents=True, exist_ok=True)
FIGS      = Path("results/figures"); FIGS.mkdir(parents=True, exist_ok=True)
SEED      = cfg["seed"]
TODAY     = "2026-10-25"


In [ ]:
data, spec = fe.load_feature_table(PROCESSED)
idx  = pd.read_parquet(PROCESSED / "dataset_index.parquet")

TRAIN = idx["split"].eq("train")
TEST  = idx["split"].eq("test")

FEATURES_GLOBAL = spec["features"]
VS_COLS = [c for c in FEATURES_GLOBAL if c.endswith(VS_CREATOR_SUFFIX)]
FEATURES_PC = [c for c in FEATURES_GLOBAL if not c.endswith(VS_CREATOR_SUFFIX)]

X_all_pc = data[FEATURES_PC].copy()
y_all    = idx["target_rel"].copy()
auth     = idx["author_id"].copy()

X_train_pc = X_all_pc[TRAIN.values].reset_index(drop=True)
y_train    = y_all[TRAIN.values].reset_index(drop=True)
auth_train = auth[TRAIN.values].reset_index(drop=True)

X_test_pc  = X_all_pc[TEST.values].reset_index(drop=True)
y_test     = y_all[TEST.values].reset_index(drop=True)
auth_test  = auth[TEST.values].reset_index(drop=True)

print(f"Per-creator features: {len(FEATURES_PC)}  (dropped *_vs_creator: {len(VS_COLS)})")
print(f"Train: {len(X_train_pc):,}  Test: {len(X_test_pc):,}  Creators: {auth_train.nunique()}")


In [ ]:
# Load B2 global XGB for baseline comparison
from creator_analytics.models import TrainedModel
b2_model = TrainedModel.load("models/B2_global_xgb.joblib")

# Load B2 test predictions from NB04
nb04_preds = pd.read_parquet(PREDS_DIR / "nb04_predictions.parquet")
b2_preds = nb04_preds["B2_XGB"].values
zero_preds = np.zeros(len(y_test))

# B2 uses global features (with vs_creator); we need to recreate those preds
# b2_preds already computed in NB04 — just reload
print("B2 baseline RMSE:", round(compute_metrics(y_test, b2_preds)["rmse"], 4))


## Config search (100 sampled creators, CV on train only)

In [ ]:
t0 = time.time()

best_rf_cfg, best_xgb_cfg = search_per_creator_config(
    X_train_pc, y_train, auth_train,
    sample_n=100,
    seed=SEED,
    n_jobs=-1,
)

elapsed_search = time.time() - t0
print(f"Config search: {elapsed_search/60:.1f} min")
print(f"Best RF  config: {best_rf_cfg}")
print(f"Best XGB config: {best_xgb_cfg}")


## B3 — Per-creator RF + B4 — Per-creator XGB

In [ ]:
t0 = time.time()

rf_models, xgb_models = train_per_creator_models(
    X_train_pc, y_train, auth_train,
    rf_config=best_rf_cfg,
    xgb_config=best_xgb_cfg,
    n_jobs=-1,
    seed=SEED,
)

elapsed_pc = time.time() - t0
print(f"Per-creator training: {elapsed_pc/60:.1f} min")

n_worse_rf  = sum(m.worse_than_zero for m in rf_models.values())
n_worse_xgb = sum(m.worse_than_zero for m in xgb_models.values())
print(f"RF  worse than zero: {n_worse_rf}/{len(rf_models)}")
print(f"XGB worse than zero: {n_worse_xgb}/{len(xgb_models)}")


## Evaluation on held-out test set

In [ ]:
rf_preds  = predict_per_creator(X_test_pc, auth_test, rf_models)
xgb_preds = predict_per_creator(X_test_pc, auth_test, xgb_models)

print(f"NaN in RF  preds: {np.isnan(rf_preds).sum()}")
print(f"NaN in XGB preds: {np.isnan(xgb_preds).sum()}")

metrics = {}
for name, preds in [
    ("B2_global_XGB",   b2_preds),
    ("B3_perCreator_RF", rf_preds),
    ("B4_perCreator_XGB", xgb_preds),
]:
    metrics[name] = compute_metrics(y_test, preds)

df_metrics = pd.DataFrame(metrics).T[["rmse","mae","r2","spearman","n"]]
display(df_metrics.round(4))


In [ ]:
# Paired comparisons vs B2 and vs predict-0
print("=== vs predict-0 ===")
for name, preds in [("B3_RF_pc", rf_preds), ("B4_XGB_pc", xgb_preds)]:
    cmp = paired_comparison(y_test, preds, zero_preds, auth_test,
                            name_a=name, name_b="predict-0")
    print(f"{name:15s}  RMSE_diff={cmp['rmse_diff']:+.4f}  "
          f"p={cmp['wilcoxon_pvalue']:.4f}  win={cmp['win_rate_a']:.3f}")

print()
print("=== vs B2 global XGB ===")
for name, preds in [("B3_RF_pc", rf_preds), ("B4_XGB_pc", xgb_preds)]:
    cmp = paired_comparison(y_test, preds, b2_preds, auth_test,
                            name_a=name, name_b="B2_XGB")
    print(f"{name:15s}  RMSE_diff={cmp['rmse_diff']:+.4f}  "
          f"p={cmp['wilcoxon_pvalue']:.4f}  win={cmp['win_rate_a']:.3f}")
    print(f"  Stratified win-rate vs B2: {cmp['stratified_win_rate']}")


In [ ]:
# Per-creator breakdown
pc_b3 = compute_per_creator_metrics(y_test, pd.Series(rf_preds,  index=y_test.index), auth_test)
pc_b4 = compute_per_creator_metrics(y_test, pd.Series(xgb_preds, index=y_test.index), auth_test)
pc_b2 = compute_per_creator_metrics(y_test, pd.Series(b2_preds,  index=y_test.index), auth_test)

print("Per-creator median RMSE:")
print(f"  B2 global XGB : {pc_b2['rmse'].median():.4f}")
print(f"  B3 per-cr RF  : {pc_b3['rmse'].median():.4f}")
print(f"  B4 per-cr XGB : {pc_b4['rmse'].median():.4f}")

print("\nPer-creator median Spearman:")
print(f"  B2 : {pc_b2['spearman'].median():.4f}")
print(f"  B3 : {pc_b3['spearman'].median():.4f}")
print(f"  B4 : {pc_b4['spearman'].median():.4f}")

# Distribution of per-creator RMSE improvement B4 vs B2
delta = pc_b4.set_index("author_id")["rmse"] - pc_b2.set_index("author_id")["rmse"]
print(f"\nB4 per-creator RMSE improvement vs B2: {(delta < 0).mean():.1%} creators improved")


In [ ]:
# Plot: per-creator RMSE distribution B2 vs B4
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pc_b2["rmse"], bins=50, alpha=0.6, label="B2 global XGB", color="steelblue")
ax.hist(pc_b4["rmse"], bins=50, alpha=0.6, label="B4 per-creator XGB", color="darkorange")
ax.set_xlabel("Per-creator RMSE")
ax.set_ylabel("Count of creators")
ax.set_title("Distribution of per-creator RMSE: global vs per-creator XGB")
ax.legend()
fig.tight_layout()
fig.savefig(FIGS / "nb05_per_creator_rmse_dist.png", dpi=120, bbox_inches="tight")
plt.show()


## Leakage check

In [ ]:
print("Leakage check — shuffle target, R² on test should be ≈ 0")
for name, predict_fn in [
    ("B3_RF_pc",  lambda X: predict_per_creator(X, auth_test, rf_models)),
    ("B4_XGB_pc", lambda X: predict_per_creator(X, auth_test, xgb_models)),
]:
    lk = leakage_check(predict_fn, X_test_pc, y_test, seed=SEED)
    print(f"{name}: R²_real={lk['r2_real']:.4f}  "
          f"R²_shuffled_mean={lk['r2_shuffled_mean']:.4f}  "
          f"(±{lk['r2_shuffled_std']:.4f})")


In [ ]:
# Save predictions + models
pd.DataFrame({
    "video_id":  idx.loc[TEST, "video_id"].values,
    "author_id": auth_test.values,
    "y_true":    y_test.values,
    "B3_RF_pc":  rf_preds,
    "B4_XGB_pc": xgb_preds,
}).to_parquet(PREDS_DIR / "nb05_predictions.parquet", index=False)

import joblib
Path("models/per_creator").mkdir(parents=True, exist_ok=True)
joblib.dump(rf_models,  "models/per_creator/B3_rf_models.joblib")
joblib.dump(xgb_models, "models/per_creator/B4_xgb_models.joblib")
print("Saved predictions and models")


In [ ]:
exp_path = Path(cfg["paths"]["experiments_log"])
for name, preds, t_s, model_name in [
    ("B3_perCreator_RF",  rf_preds,  elapsed_pc,
     f"RF per-creator (RF cfg: {best_rf_cfg})"),
    ("B4_perCreator_XGB", xgb_preds, elapsed_pc,
     f"XGB per-creator (XGB cfg: {best_xgb_cfg})"),
]:
    cmp_b2 = paired_comparison(y_test, preds, b2_preds, auth_test, name, "B2_XGB")
    log_experiment(
        exp_path, name=name, date=TODAY,
        features=f"{len(FEATURES_PC)} per-creator (no *_vs_creator)",
        model=model_name,
        params={},
        cv_rmse=None,
        test_metrics=compute_metrics(y_test, preds),
        train_time_s=t_s,
        comparison=cmp_b2,
        comment="NB05; vs B2 global XGB",
    )
print("Logged experiments")


## Summary

| Model | Test RMSE | vs B2 RMSE diff | p-value | Win rate vs B2 |
|-------|-----------|-----------------|---------|----------------|
| B2 global XGB | baseline | — | — | — |
| B3 per-creator RF | — | — | — | — |
| B4 per-creator XGB | — | — | — | — |

*(Filled from cell output)*

**Key findings:**
- Per-creator models are expected to improve for creators with enough data
- Small creators may regress (not enough data to learn per-creator patterns)
- C3 shrinkage blend (NB06) will combine both using `w = n/(n+k)`